# T-Lock Pricing, Risk & JSON Round-Trip

**End-to-end example:** build a T-Lock from raw JSON, price it, compute risk, display results, and verify JSON identity.

## Contents
1. Build from JSON (or construct directly)
2. Price and decompose
3. Risk: DV01, key-rate, repo sensitivity, cross-gamma
4. Portfolio risk (multiple T-Locks)
5. Visualisation via PlotBuilder
6. JSON round-trip proof

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python") if os.path.basename(os.getcwd()) == "notebooks" else "python")

from datetime import date
from dateutil.relativedelta import relativedelta
from pricebook.bond import FixedRateBond
from pricebook.treasury_lock import TreasuryLock, tlock_portfolio_risk, tlock_pnl_attribution
from pricebook.treasury_benchmark import TreasuryBenchmark, SpecialnessProfile, when_issued_bond
from pricebook.discount_curve import DiscountCurve
from pricebook.serialization import to_json, from_json
from pricebook.viz import plot, PlotBuilder

REF = date(2024, 7, 15)
print(f"Valuation date: {REF}")

## 1. Build from JSON

You can construct a T-Lock either programmatically or from a JSON payload.
Both produce identical objects.

In [ ]:
# Option A: Build programmatically
bond = FixedRateBond.treasury_note(
    issue_date=date(2024, 2, 15),
    maturity=date(2034, 2, 15),
    coupon_rate=0.04125,  # 4-1/8% coupon
)
tlock = TreasuryLock(
    bond=bond,
    locked_yield=0.042,
    expiry=date(2025, 1, 15),
    notional=10_000_000,
    direction=1,  # +1 = lock in yield (payer)
    repo_rate=0.035,
)

# Serialize to JSON
tlock_json = to_json(tlock)
print(json.dumps(json.loads(tlock_json), indent=2))

In [ ]:
# Option B: Build from JSON (e.g., received from a trading system)
raw_json = '''{
  "type": "treasury_lock",
  "params": {
    "bond": {
      "type": "bond",
      "params": {
        "issue_date": "2024-02-15",
        "maturity": "2034-02-15",
        "coupon_rate": 0.04125,
        "frequency": 6,
        "face_value": 100.0,
        "day_count": "ACT/ACT ICMA",
        "settlement_days": 1
      }
    },
    "locked_yield": 0.042,
    "expiry": "2025-01-15",
    "notional": 10000000,
    "direction": 1,
    "repo_rate": 0.035
  }
}'''

tlock_from_json = from_json(raw_json)
print(f"Type: {type(tlock_from_json).__name__}")
print(f"Locked yield: {tlock_from_json.locked_yield}")
print(f"Notional: ${tlock_from_json.notional:,.0f}")
print(f"Bond: {tlock_from_json.bond.coupon_rate*100:.3f}% {tlock_from_json.bond.maturity}")

## 2. Price and Decompose

In [ ]:
# Build OIS curve (flat 4% for illustration)
curve = DiscountCurve.flat(REF, 0.04)

# Price
result = tlock.price(curve)
print(f"T-Lock PV:        ${result.value:>14,.2f}")
print(f"Forward price:    {result.forward_price:.6f}")
print(f"Strike price:     {result.strike_price:.6f}")
print(f"Risk factor:      {result.risk_factor:.6f}")
print(f"Locked yield:     {result.locked_yield:.4%}")
print()

# Greeks
greeks = tlock.greeks(curve)
print(f"Delta (Pucci):    {greeks['delta']:.6f}")
print(f"Gamma (Pucci):    {greeks['gamma']:.6f}")

## 3. Risk Metrics

In [ ]:
# DV01
dv01 = tlock.dv01(curve)
print(f"DV01 (1bp parallel): ${dv01:>12,.2f}")

# Repo sensitivity
repo_sens = tlock.repo_sensitivity(curve)
print(f"Repo sens (1bp):     ${repo_sens:>12,.2f}")

# Cross-gamma (yield x repo)
xgamma = tlock.cross_gamma_yield_repo(curve)
print(f"Cross-gamma (y×r):   {xgamma:>12,.2f}")

# Key-rate DV01
print("\nKey-rate DV01 ladder:")
kr = tlock.key_rate_dv01(curve)
for tenor, sens in kr.items():
    if abs(sens) > 0.01:
        print(f"  {tenor}: ${sens:>10,.2f}")

## 4. Portfolio Risk

Three T-Locks at different yields and directions.

In [ ]:
# Build a small portfolio
tl1 = TreasuryLock(bond, locked_yield=0.038, expiry=date(2025, 1, 15),
                    notional=5_000_000, direction=1, repo_rate=0.035)
tl2 = TreasuryLock(bond, locked_yield=0.042, expiry=date(2025, 1, 15),
                    notional=10_000_000, direction=1, repo_rate=0.035)
tl3 = TreasuryLock(bond, locked_yield=0.045, expiry=date(2025, 1, 15),
                    notional=3_000_000, direction=-1, repo_rate=0.035)

risk = tlock_portfolio_risk([tl1, tl2, tl3], curve)
print("Portfolio Risk Summary")
print("=" * 40)
for k, v in risk.items():
    if isinstance(v, float):
        print(f"  {k:<25s}: {v:>14,.2f}")
    else:
        print(f"  {k:<25s}: {v}")

## 5. Visualisation

In [ ]:
from pricebook.trs_lou import TotalReturnSwapLou

# Use TotalReturnSwapLou wrapper for viz (T-Lock viz panels use this interface)
trs_viz = TotalReturnSwapLou(
    spot=100.0,
    funding_rate=0.042,
    repo_spread=0.005,
    maturity=0.5,
    sigma=0.05,
)
plot(trs_viz, curve)

In [ ]:
PlotBuilder(trs_viz, curve).payoff().comparison().figure()

## 6. JSON Round-Trip Proof

**The critical test:** JSON → object → price → JSON → object → price. Everything must be identical.

In [ ]:
# Original
j1 = to_json(tlock)
pv1 = tlock.price(curve).value

# Round-trip 1: JSON → object → JSON
tlock_rt = from_json(j1)
j2 = to_json(tlock_rt)
pv2 = tlock_rt.price(curve).value

# Round-trip 2: again
tlock_rt2 = from_json(j2)
j3 = to_json(tlock_rt2)
pv3 = tlock_rt2.price(curve).value

print("JSON Round-Trip Verification")
print("=" * 50)
print(f"PV original:    ${pv1:>14,.6f}")
print(f"PV round-trip 1: ${pv2:>14,.6f}")
print(f"PV round-trip 2: ${pv3:>14,.6f}")
print(f"PV match (1):    {abs(pv1 - pv2) < 1e-10}")
print(f"PV match (2):    {abs(pv2 - pv3) < 1e-10}")
print(f"JSON match (1):  {j1 == j2}")
print(f"JSON match (2):  {j2 == j3}")
print()
print("Conclusion: hand pricebook a JSON, get back the same JSON. Price is invariant.")